In [ ]:
import pandas as pd
import numpy as np
import json
import logging
from typing import List, Tuple, Dict, Any, Optional
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
import warnings
import torch
warnings.filterwarnings('ignore')

In [ ]:
file_path = '../data/efsa_sentiment_classification.csv'
df = pd.read_csv(file_path)

df = df[:10]

In [ ]:
# MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
MODEL_NAME = "FinGPT/fingpt-sentiment_llama2-13b_lora"

SENTIMENT_POLARITIES = ['POS', 'NEG', 'NEU']
BATCH_SIZE = 10

FINANCIAL_EVENT_LIST = {
    "Financial Affairs": ["Profit Announcement", "Profit Forecast", "Other Financial Affairs"],
    "Shareholder Affairs": ["Stock Holding Adjustment", "Shareholder Pledge", "Release of Pledge", "Other Shareholder Affairs"],
    "Stock Affairs": ["Stock Price Movement", "Equity Incentives & Employee Stock Ownership Plans", "Stock Dividend", "Stock Buyback", "Stock Status", "Restricted Shares Release", "Other Stock Affairs"],
    "Business Operations": ["Product Dynamics", "Capacity Changes", "Initiating Cooperation", "Technical Quality Control, Qualification Changes", "Government Subsidies", "New Company Establishment", "Institutional Research", "Intellectual Property", "Sales, Market Share Changes", "Project Bidding", "Project Dynamics", "Other Business Operations Affairs"],
    "Compliance and Credit": ["Company Litigation", "Rating Adjustment", "Legal Affairs", "Clarification Announcements", "Regulatory Inquiries", "Case Investigations", "Administrative Penalties", "Other Compliance and Credit Affairs"],
    "Management Affairs": ["Employee Dynamics", "Directors, Supervisors, and Senior Executives Dynamics"],
    "Financing and Investment": ["Company Listing", "Mergers and Acquisitions", "Investment Events", "Stock Issuance", "Financing and Margin Trading", "Capital Flows", "Other Financing and Investment Affairs"]
}

FINE_TO_COARSE = {}
FINE_EVENT_LIST = []

for coarse_key, fine_list in FINANCIAL_EVENT_LIST.items():
    for fine_event in fine_list:
        FINE_TO_COARSE[fine_event] = coarse_key
        FINE_EVENT_LIST.append(fine_event)

INDUSTRY_LIST = df['Industry'].unique().tolist()

coarse_event_definitions = """
- Financial Affairs: Earnings announcements, forecasts, or other financial updates.
- Shareholder Affairs: Actions by shareholders such as pledges, stock adjustments.
- Stock Affairs: Stock buybacks, dividends, price movement announcements.
- Business Operations: New products, partnerships, quality control, or business activities.
- Compliance and Credit: Legal issues, investigations, rating changes.
- Management Affairs: Executive hires, promotions, or organizational changes.
- Financing and Investment: Mergers, acquisitions, funding rounds, IPOs.
"""

In [ ]:
print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", trust_remote_code=True).half()
print("Model loaded successfully!")

In [ ]:
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, LlamaForCausalLM, LlamaTokenizerFast
from peft import PeftModel

print("Loading model and tokenizer...")
base_model = "NousResearch/Llama-2-13b-hf"
peft_model = "FinGPT/fingpt-sentiment_llama2-13b_lora"
tokenizer = LlamaTokenizerFast.from_pretrained(
    base_model, trust_remote_code=True)
model = LlamaForCausalLM.from_pretrained(
    base_model, trust_remote_code=True, device_map="cuda:0", load_in_8bit=True,)
model = PeftModel.from_pretrained(model, peft_model)
model = model.eval()
print("Model loaded successfully!")

In [ ]:
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
logger = logging.getLogger(__name__)


def query_model(model, model_name, tokenizer, prompt: str, max_new_tokens=512, temperature=0.7) -> str:
    """Generate model response for given prompt using Mistral Instruct format."""

    if 'mistral' in model_name:
        instruction = f"[INST] {prompt} [/INST]"
    elif 'llama' in model_name:
        instruction = (
            f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n"
            f"{prompt}\n"
            f"<|start_header_id|>assistant<|end_header_id|>\n"
        )

    inputs = tokenizer(instruction, return_tensors="pt").to(model.device)
    try:
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                                    temperature=temperature, pad_token_id=tokenizer.eos_token_id)
        response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        # Remove prompt from response to get just the answer:
        answer = response.split(prompt)[-1].strip()
        return answer
    except Exception as e:
        logger.error(f"Error querying model: {e}")
        return ""


def extract_companies(text: str) -> str:
    """Extract company names from financial text."""
    prompt = (f"Financial text: {text}\n\n"
              "As a financial analyst, identify ALL company names mentioned in this text. "
              "If multiple companies are mentioned, separate them with commas. "
              "If only one company is mentioned, just provide that company name. "
              "Only answer with unique company names in full, not names of individuals, without additional text or punctuation."
              )

    try:
        response = query_model(
            model, MODEL_NAME, tokenizer, prompt, temperature=0.2)
        companies = [company.strip()
                     for company in response.split(',') if company.strip()]
        return companies if companies else ["Unknown Company"]
    except Exception as e:
        logger.error(f"Error extracting company name: {e}")
        return "Unknown Company"


def get_industry(text: str, company_name: str) -> str:
    """Get industry for a company using model prompt."""
    prompt = (f"You are a financial analyst using professional databases to classify companies. "
              f"Text context: {text}\n"
              f"Company: {company_name}\n\n"
              f"As if consulting Bloomberg Terminal, FactSet, or Yahoo Finance, determine {company_name}'s industry.\n\n"
              # f"Analysis framework:\n"
              # f"1. Primary Business: What does {company_name} mainly do for revenue?\n"
              # f"2. Database Classification: How would major financial platforms categorize this company?\n"
              f"3. GICS Mapping: Which GICS sector best fits?\n\n"
              f"Available GICS sectors: {INDUSTRY_LIST}\n\n"
              # f"Key considerations:\n"
              # f"- Primary revenue source\n"
              # f"- Standard financial database classifications\n"
              # f"- How institutional investors would categorize this stock\n"
              # f"- SEC filing industry classifications\n\n"
              f"Based on professional financial database standards, select the most appropriate GICS sector.\n"
              "Respond only with the industry name from the list, without additional text or punctuation."
              )

    try:
        response = query_model(
            model, MODEL_NAME, tokenizer, prompt, temperature=0.4)
        response = response.replace('"', '').replace("'", "")
        return response.strip() or "Unknown Industry"
    except Exception as e:
        logger.error(f"Error getting industry for {company_name}: {e}")
        return "Unknown Industry"


def classify_coarse_grained_event(text: str, industry: str, company_name: str) -> Tuple[str, Any]:
    """Classify the coarse-grained event type."""
    coarse_events = list(FINANCIAL_EVENT_LIST.keys())
    prompt = (f"You are a financial sentiment model.\n\n"
              f"Text: \"{text}\"\n"
              f"Company: {company_name}\n"
              f"Industry: {industry}\n\n"
              f"Choose the most appropriate coarse-grained event type from this list based on what is explicitly stated in the text: {coarse_events}\n\n"
              f"The following are the event definitions: {coarse_event_definitions}"
              "Please only answer with one coarse-grained event type, without additional text or punctuation."
              )

    try:
        response = query_model(
            model, MODEL_NAME, tokenizer, prompt, temperature=0.1)
        response = response.replace('"', '').replace("'", "")
        return response.strip() or "Unknown Event"
    except Exception as e:
        logger.error(f"Error classifying coarse event: {e}")
        return "Unknown Event"


def classify_fine_grained_event(text: str, industry: str, company_name: str, coarse_event: str) -> Tuple[str, Any]:
    """Classify the fine-grained event type."""
    if coarse_event in FINANCIAL_EVENT_LIST:
        fine_events = FINANCIAL_EVENT_LIST[coarse_event]
        prompt = (f"You are a financial sentiment model. The following news text was classified as the event category: {coarse_event}.\n\n"
                  f"Text: \"{text}\"\n"
                  f"Company: {company_name}\n"
                  f"Industry: {industry}\n\n"
                  f"Now select the specific fine-grained event that best describes what happened, based on this list {fine_events}.\n\n"
                  "If the event doesn't fit perfectly into a specific event type, choose the most relevant 'Other' category.\n"
                  "Please only answer with the fine-grained event type, without additional text or punctuation."
                  )
    else:
        return "Unknown Fine Event"

    try:
        response = query_model(
            model, MODEL_NAME, tokenizer, prompt, temperature=0.3)
        response = response.replace('"', '').replace("'", "")
        return response.strip() or "Unknown Fine Event"
    except Exception as e:
        return "Unknown Fine Event"


def classify_fine_grained_event_first(text: str, industry: str, company_name: str) -> Tuple[str, str, Any]:
    """Classify fine-grained event first, then map to coarse."""

    prompt = (f"Financial text: {text}\n"
              f"Company: {company_name}\n"
              f"Industry: {industry}\n\n"
              f"As a financial sentiment model, determine the specific type of corporate event that occurred at {company_name}. "
              f"Select the most precise event type from this comprehensive list: {FINE_EVENT_LIST}\n\n"
              "If none of the event types are explicity mentioned in the text, choose the most relevant 'Other' category.\n"
              "Respond only with an event type from the provided list, without additional text, quotation marks, or punctuation."
              )

    try:
        fine_response = query_model(
            model, MODEL_NAME, tokenizer, prompt, temperature=0.5).strip()

        # Clean quotation marks
        fine_response = fine_response.replace('"', '').replace("'", "")

        # Map to coarse event
        coarse_event = FINE_TO_COARSE.get(fine_response, "Unknown Event")

        return coarse_event, fine_response
    except Exception as e:
        logger.error(f"Error classifying fine-grained event first: {e}")
        return "Unknown Event", "Unknown Fine Event"


def analyze_sentiment(text: str, industry: str, company_name: str, coarse_event: str, fine_event: str) -> str:
    """Analyze sentiment polarity of the event."""
    prompt = (f"You are a financial sentiment model. Analyze the sentiment expressed in the text.\n\n"
              f"Company: {company_name} ({industry})\n"
              f"Event: {fine_event}\n"
              f"News: {text}\n\n"
              f"- POS = The sentence includes positive or optimistic language.\n"
              f"- NEG = The sentence includes negative or pessimistic language.\n"
              "Please only answer with one of: POS, NEU, or NEG, without additional text or punctuation."
              )

    try:
        response = query_model(
            model, MODEL_NAME, tokenizer, prompt, temperature=0.5)
        return response.strip() or "Unknown"
    except Exception as e:
        logger.error(f"Error analyzing sentiment: {e}")
        return "Unknown"

In [ ]:
def analyze_financial_text(text: str) -> Tuple[str, str, str, str, str]:
    """
    Analyze a single financial text and return the quintuple.

    Returns:
        Tuple[str, str, str, str, str]: (company_name, industry, coarse_event, fine_event, sentiment)
    """
    try:
        # Extract all company names
        company_names = extract_companies(text)
        results = []

        for company_name in company_names:
            try:
                # Get industry
                industry = get_industry(text, company_name)

                # Classify fine-grained event first, then derive coarse event
                coarse_event = classify_coarse_grained_event(
                    text, company_name, industry)

                fine_event = classify_fine_grained_event(
                    text, industry, company_name, coarse_event)

                # Analyze sentiment
                sentiment = analyze_sentiment(
                    text, industry, company_name, coarse_event, fine_event)

                results.append((text, company_name, industry,
                               coarse_event, fine_event, sentiment))

            except Exception as e:
                logger.error(
                    f"Error analyzing text for company {company_name}: {e}")
                results.append(
                    (company_name, "Error", "Error", "Error", "Error"))

        return results

    except Exception as e:
        logger.error(f"Error analyzing text: {e}")
        return [("Error", "Error", "Error", "Error", "Error")]

In [ ]:
results = []
# df = df[:100]
# Process each text
for idx, row in df.iterrows():
    text = row['Sentence']

    if pd.isna(text) or text.strip() == '':
        logger.warning(f"Empty text at index {idx}")
        results.append(("", "", "", "", "", ""))
        continue

    print(f"Processing {idx+1}/{len(df)}: {text[:50]}...")

    # Analyze the text
    result = analyze_financial_text(text)

    # Store results
    results.append(result)

    # Print progress every 10 items
    if (idx + 1) % 10 == 0:
        print(f"Completed {idx + 1}/{len(df)} texts")

In [ ]:
# Flatten the list of lists and add the original sentence to each entry
flat_results_with_sentence = []
for i, sentence_results in enumerate(results):
    for company_result in sentence_results:
        flat_results_with_sentence.append(company_result)

results_df = pd.DataFrame(flat_results_with_sentence, columns=[
                          'Sentence', 'Company', 'Industry', 'Coarse Event', 'Fine Event', 'Sentiment'])
results_df['Sentiment'] = results_df['Sentiment'].str[:3]

In [ ]:
results_df.head(20)

In [ ]:
df[:30]

In [ ]:
from collections import defaultdict
from sklearn.metrics import f1_score


def calculate_f1_by_label(df, results_df):
    preds_by_sentence = results_df.groupby('Sentence')
    cols = ['Sentiment', 'Industry',
            'Coarse-Grained Event', 'Fine-Grained Event']

    gt_labels = defaultdict(list)
    pred_labels = defaultdict(list)

    for _, row in df.iterrows():
        sentence = row['Sentence']
        gt_company = str(row['Company']).lower()

        if sentence not in preds_by_sentence.groups:
            for col in cols:
                gt_labels[col].append(str(row[col]))
                pred_labels[col].append('NO_PREDICTION')
            continue

        preds = preds_by_sentence.get_group(sentence)

        # match if either one contains the other
        company_match = company_match = preds[preds['Company'].str.lower().apply(
            lambda x: gt_company in x or x in gt_company)]

        if company_match.empty:
            for col in cols:
                gt_labels[col].append(str(row[col]))
                pred_labels[col].append('NO_COMPANY_MATCH')
            continue

        pred_row = company_match.iloc[0]

        for col in cols:
            gt_val = str(row[col])
            pred_val = str(pred_row[col])
            gt_labels[col].append(gt_val)
            pred_labels[col].append(pred_val)

    # Compute F1 scores
    results = {}
    for col in cols:
        lab_true = gt_labels[col]
        lab_pred = pred_labels[col]
        labels = list(set(lab_true + lab_pred))

        macro_f1 = f1_score(lab_true, lab_pred, average='macro',
                            labels=labels, zero_division=0)
        per_label_f1 = f1_score(
            lab_true, lab_pred, average=None, labels=labels, zero_division=0)

        results[col] = {
            'f1_macro': round(macro_f1, 4),
            'f1_per_label': {label: round(score, 4) for label, score in zip(labels, per_label_f1)}
        }

    return results

In [ ]:
from sklearn.metrics import f1_score
from collections import defaultdict


def calculate_efsa_variant_macro_f1(df, results_df):
    variants = {
        'EFSA': ['Company', 'Industry', 'Coarse-Grained Event', 'Fine-Grained Event', 'Sentiment'],
        'Coarse-grained EFSA': ['Company', 'Industry', 'Coarse-Grained Event', 'Sentiment'],
        'Fine-grained EFSA': ['Company', 'Industry', 'Fine-Grained Event', 'Sentiment'],
        'Entity-Level FSA': ['Company', 'Industry', 'Sentiment']
    }

    preds_by_sentence = results_df.groupby('Sentence')

    lab_true = defaultdict(list)
    lab_pred = defaultdict(list)

    for _, row in df.iterrows():
        sentence = row['Sentence']
        gt_company = str(row['Company']).lower()

        if sentence not in preds_by_sentence.groups:
            for variant, cols in variants.items():
                lab_true[variant].append(
                    "|".join(str(row[col]) for col in cols))
                lab_pred[variant].append(
                    "|".join(['NO_PREDICTION'] * len(cols)))
            continue

        preds = preds_by_sentence.get_group(sentence)

        company_match = preds[preds['Company'].str.lower().apply(
            lambda x: gt_company in x or x in gt_company)]

        if company_match.empty:
            for variant, cols in variants.items():
                lab_true[variant].append(
                    "|".join(str(row[col]) for col in cols))
                lab_pred[variant].append(
                    "|".join(['NO_COMPANY_MATCH'] * len(cols)))
            continue

        pred_row = company_match.iloc[0]

        for variant, cols in variants.items():
            gt_string = "|".join(str(row[col]) for col in cols)
            pred_string = "|".join(str(pred_row[col]) for col in cols)
            lab_true[variant].append(gt_string)
            lab_pred[variant].append(pred_string)

    results = {}
    for variant in variants:
        true = lab_true[variant]
        pred = lab_pred[variant]
        labels = list(set(true + pred))
        macro_f1 = f1_score(true, pred, average='macro',
                            labels=labels, zero_division=0)
        results[variant] = round(macro_f1, 4)

    return results